# FENRIX Multi-Company Anonymization Pipeline

This notebook orchestrates the FENRIX pipeline to collect, anonymize, and QA financial data for multiple companies.

**Outputs:**
- Structured originals (OHLCV, financial statements)
- SEC filings and normalized text
- News articles
- Deterministic anonymized counterparts
- Manifests, coverage reports, residual QA reports
- Export ZIP (anonymized only)

## 1. Repository Setup

In [ ]:
# Clone or select repository branch
import os
import sys

REPO_URL = "https://github.com/Scott-Switzer/fenrix-synthetic-data.git"
BRANCH = "feature/colab-multicompany-anonymization"

if not os.path.exists("/content/fenrix-synthetic-data"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    %cd /content/fenrix-synthetic-data
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd /content/fenrix-synthetic-data

## 2. Install Dependencies

In [ ]:
# Install package with pinned dependencies
!pip install -e .
!pip install yfinance

## 3. Optional: Mount Google Drive

In [ ]:
# Uncomment to mount Google Drive for persistent output
# from google.colab import drive
# drive.mount('/content/drive')
# OUTPUT_ROOT = "/content/drive/MyDrive/fenrix_output"

## 4. Parameters

In [ ]:
# Pipeline parameters
TICKER = "NVDA"  # Single ticker mode
# COMPANIES_CSV = "configs/companies.csv"  # Batch mode (uncomment to use)
YEARS = 10
OUTPUT_ROOT = "/content/fenrix_output"
ENABLE_NVIDIA = False
COLLECT_ONLY = False
ANONYMIZE_ONLY = False
DRY_RUN = False

# SEC configuration
SEC_USER_AGENT = os.environ.get("SEC_USER_AGENT", "")

print(f"Ticker: {TICKER}")
print(f"Years: {YEARS}")
print(f"Output: {OUTPUT_ROOT}")
print(f"NVIDIA: {ENABLE_NVIDIA}")

## 5. Environment Validation

In [ ]:
import subprocess

# Verify package installation
result = subprocess.run(["fenrix-synth", "--version"], capture_output=True, text=True)
print(result.stdout or result.stderr)

# Check yfinance
try:
    import yfinance
    print(f"yfinance: {yfinance.__version__}")
except ImportError:
    print("yfinance not installed")

# Check output directory
import os
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f"Output directory ready: {OUTPUT_ROOT}")

## 6. Dry-Run Preview

In [ ]:
if DRY_RUN:
    !fenrix-synth pipeline-run \
        --ticker {TICKER} \
        --years {YEARS} \
        --output-root {OUTPUT_ROOT} \
        --dry-run
else:
    print("Dry-run disabled. Set DRY_RUN = True to preview.")

## 7. Execute Pipeline

In [ ]:
cmd = f"""fenrix-synth pipeline-run \
    --ticker {TICKER} \
    --years {YEARS} \
    --output-root {OUTPUT_ROOT}"""

if ENABLE_NVIDIA:
    cmd += " \
    --enable-nvidia"
if COLLECT_ONLY:
    cmd += " \
    --collect-only"
if ANONYMIZE_ONLY:
    cmd += " \
    --anonymize-only"
if SEC_USER_AGENT:
    cmd += f" \
    --sec-user-agent '{SEC_USER_AGENT}'"

print(f"Running: {cmd}")
!{cmd}

## 8. Display Run Summary

In [ ]:
import json
from pathlib import Path

run_dirs = sorted(Path(OUTPUT_ROOT).glob("run_*"))
if run_dirs:
    latest_run = run_dirs[-1]
    summary_path = latest_run / "run_summary.json"
    if summary_path.exists():
        with open(summary_path) as f:
            summary = json.load(f)
        print(json.dumps(summary, indent=2))
    else:
        print(f"No summary found in {latest_run}")
else:
    print("No runs found.")

## 9. Preview Artifacts

In [ ]:
import pandas as pd

if run_dirs:
    run_dir = run_dirs[-1]
    ticker_dir = run_dir / "originals" / TICKER
    anon_dir = run_dir / "anonymized" / TICKER

    # Original metrics preview
    ohlcv_path = ticker_dir / "metrics" / "ohlcv.parquet"
    if ohlcv_path.exists():
        df = pd.read_parquet(ohlcv_path)
        print("=== Original OHLCV (first 5 rows) ===")
        display(df.head())
        print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
        print(f"Date range: {df.index.min()} to {df.index.max()}")

    # Anonymized features preview
    features_path = anon_dir / "metrics" / "features_s3a.json"
    if features_path.exists():
        with open(features_path) as f:
            features_data = json.load(f)
        print("\n=== Anonymized Features (first 3 rows) ===")
        for row in features_data.get("features", [])[:3]:
            print(row)
        print(f"Total feature rows: {features_data.get('row_count', 0)}")

    # Filing excerpt
    filings_dir = ticker_dir / "sec" / "filings"
    md_files = list(filings_dir.glob("*.md")) if filings_dir.exists() else []
    if md_files:
        text = md_files[0].read_text()[:500]
        print("\n=== Original Filing Excerpt ===")
        print(text[:500])

    anon_filings_dir = anon_dir / "sec"
    anon_md_files = list(anon_filings_dir.glob("*.md")) if anon_filings_dir.exists() else []
    if anon_md_files:
        text = anon_md_files[0].read_text()[:500]
        print("\n=== Anonymized Filing Excerpt ===")
        print(text[:500])

    # News preview
    news_path = ticker_dir / "news" / "articles.json"
    if news_path.exists():
        with open(news_path) as f:
            articles = json.load(f)
        if articles:
            print("\n=== News Record ===")
            print(json.dumps(articles[0], indent=2))

    # Coverage report
    coverage_path = run_dir / "qa" / TICKER / "source_coverage" / "coverage_report.json"
    if coverage_path.exists():
        with open(coverage_path) as f:
            coverage = json.load(f)
        print("\n=== Coverage Report ===")
        print(json.dumps(coverage, indent=2))

    # Residual QA
    qa_path = run_dir / "qa" / TICKER / "residual_scans" / "qa_report.json"
    if qa_path.exists():
        with open(qa_path) as f:
            qa = json.load(f)
        print("\n=== Residual QA Summary ===")
        print(json.dumps(qa, indent=2))

## 10. Export ZIP

In [ ]:
import hashlib

if run_dirs:
    run_dir = run_dirs[-1]
    export_path = run_dir / "exports" / "anonymized_bundle.zip"
    if export_path.exists():
        sha256 = hashlib.sha256(export_path.read_bytes()).hexdigest()
        print(f"Export ZIP: {export_path}")
        print(f"Size: {export_path.stat().st_size / 1024 / 1024:.2f} MB")
        print(f"SHA-256: {sha256}")
    else:
        print("No export ZIP found.")
else:
    print("No runs found.")

## 11. Final Paths and Checksums

In [ ]:
if run_dirs:
    run_dir = run_dirs[-1]
    print(f"Run directory: {run_dir}")
    print("\nTop-level files:")
    for f in sorted(run_dir.rglob("*")):
        if f.is_file():
            rel = f.relative_to(run_dir)
            size = f.stat().st_size
            print(f"  {rel} ({size} bytes)")
else:
    print("No runs found.")